<a href="https://colab.research.google.com/github/pngl2415/2.6-openMP/blob/main/ACTIVIDAD_3_4_BANCO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!nvidia-smi
!nvcc --version


Mon Nov  9 08:41:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

import os

os.makedirs("/content/Practica_Banco_CUDA", exist_ok=True)
os.chdir("/content/Practica_Banco_CUDA")

print("Carpeta de trabajo:", os.getcwd())


Carpeta de trabajo: /content/Practica_Banco_CUDA


In [ ]:

%%writefile banco.cu
#include <iostream>
#include <cuda_runtime.h>

using namespace std;

// Integrantes:
// Guzman Lopez Paola Nicole
// Mondragon Gambino Juan Sinuhe
// Morales Limon Jennifer Nataly
class Banco {
public:
    int saldo;
    int mutex;

    __host__ __device__
    Banco(int saldoInicial = 0) {
        saldo = saldoInicial;
        mutex = 0;
    }

    __device__
    void bloquear() {
        while (atomicCAS(&mutex, 0, 1) != 0) {
            // Esperar hasta obtener el bloqueo
        }
        __threadfence();
    }

    __device__
    void desbloquear() {
        __threadfence();
        atomicExch(&mutex, 0);
    }

    __device__
    int depositar(int cantidad) {
        bloquear();

        atomicAdd(&saldo, cantidad);
        int nuevoSaldo = saldo;

        desbloquear();
        return nuevoSaldo;
    }

    __device__
    int retirar(int cantidad) {
        bloquear();

        int nuevoSaldo = saldo;

        if (saldo >= cantidad) {
            atomicSub(&saldo, cantidad);
            nuevoSaldo = saldo;
        }

        desbloquear();
        return nuevoSaldo;
    }

    __device__
    int consultarSaldo() {
        bloquear();

        int saldoActual = saldo;

        desbloquear();
        return saldoActual;
    }
};

int main() {
    cout << "Guzman Lopez Paola Nicole | "
         << "Mondragon Gambino Juan Sinuhe | "
         << "Morales Limon Jennifer Nataly" << endl;

    Banco* banco;

    cudaError_t error = cudaMallocManaged(&banco, sizeof(Banco));

    if (error != cudaSuccess) {
        cerr << "Error al reservar memoria: "
             << cudaGetErrorString(error) << endl;
        return 1;
    }

    new (banco) Banco(10000);

    cout << "\n====================================\n";
    cout << "   SISTEMA BANCARIO CUDA - PRUEBA\n";
    cout << "====================================\n";
    cout << "Saldo inicial: $" << banco->saldo << endl;
    cout << "Mutex inicial: " << banco->mutex << endl;

    cudaFree(banco);

    cout << "\nGuzman Lopez Paola Nicole | "
         << "Mondragon Gambino Juan Sinuhe | "
         << "Morales Limon Jennifer Nataly" << endl;
}


Writing banco.cu


In [ ]:

!nvcc -std=c++17 -arch=sm_75 banco.cu -o banco


In [ ]:

!./banco


Guzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly

   SISTEMA BANCARIO CUDA - PRUEBA
Saldo inicial: $10000
Mutex inicial: 0

Guzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly


In [ ]:


%%writefile banco.cu
#include <iostream>
#include <cstdio>
#include <new>
#include <cuda_runtime.h>

using namespace std;

// Integrantes:
// Guzman Lopez Paola Nicole
// Mondragon Gambino Juan Sinuhe
// Morales Limon Jennifer Nataly
class Banco {
public:
    int saldo;
    int mutex;

    __host__ __device__
    Banco(int saldoInicial = 0) {
        saldo = saldoInicial;
        mutex = 0;
    }

    __device__
    void bloquear() {
        while (atomicCAS(&mutex, 0, 1) != 0) {
            // Esperar hasta obtener el bloqueo
        }
        __threadfence();
    }

    __device__
    void desbloquear() {
        __threadfence();
        atomicExch(&mutex, 0);
    }

    __device__
    int depositar(int cantidad) {
        bloquear();

        atomicAdd(&saldo, cantidad);
        int nuevoSaldo = saldo;

        desbloquear();
        return nuevoSaldo;
    }

    __device__
    int retirar(int cantidad) {
        bloquear();

        int nuevoSaldo = saldo;

        if (saldo >= cantidad) {
            atomicSub(&saldo, cantidad);
            nuevoSaldo = saldo;
        }

        desbloquear();
        return nuevoSaldo;
    }

    __device__
    int consultarSaldo() {
        bloquear();

        int saldoActual = saldo;

        desbloquear();
        return saldoActual;
    }
};

// Generador sencillo de numeros pseudoaleatorios
__device__
unsigned int generarAleatorio(unsigned int semilla) {
    semilla ^= semilla << 13;
    semilla ^= semilla >> 17;
    semilla ^= semilla << 5;
    return semilla;
}

// Kernel para depositos
__global__
void depositarKernel(Banco* banco, int* montos, int* resultados) {
    int hilo = blockIdx.x * blockDim.x + threadIdx.x;
    int cantidad = montos[hilo];

    int nuevoSaldo = banco->depositar(cantidad);
    resultados[hilo] = nuevoSaldo;

    printf("[DEVICE] Bloque: %d | Hilo: %d | DEPOSITO | Monto: $%d | Nuevo saldo: $%d\n",
           blockIdx.x, threadIdx.x, cantidad, nuevoSaldo);
}

// Kernel para retiros
__global__
void retirarKernel(Banco* banco, int* montos, int* resultados) {
    int hilo = blockIdx.x * blockDim.x + threadIdx.x;
    int cantidad = montos[hilo];

    int nuevoSaldo = banco->retirar(cantidad);
    resultados[hilo] = nuevoSaldo;

    printf("[DEVICE] Bloque: %d | Hilo: %d | RETIRO | Monto: $%d | Nuevo saldo: $%d\n",
           blockIdx.x, threadIdx.x, cantidad, nuevoSaldo);
}

// Kernel para consultas
__global__
void consultarSaldoKernel(Banco* banco, int* resultados) {
    int hilo = blockIdx.x * blockDim.x + threadIdx.x;

    int saldoActual = banco->consultarSaldo();
    resultados[hilo] = saldoActual;

    printf("[DEVICE] Bloque: %d | Hilo: %d | CONSULTA | Monto: $0 | Saldo: $%d\n",
           blockIdx.x, threadIdx.x, saldoActual);
}

// Kernel que selecciona aleatoriamente la operacion
__global__
void operacionesAleatoriasKernel(Banco* banco, int* montos,
                                int* operaciones, int* resultados,
                                unsigned int semilla) {
    int hilo = blockIdx.x * blockDim.x + threadIdx.x;

    unsigned int aleatorio =
        generarAleatorio(semilla ^ (hilo + 1) * 747796405u);

    int operacion = aleatorio % 3;
    aleatorio = generarAleatorio(aleatorio);
    int cantidad = (aleatorio % 500) + 1;

    operaciones[hilo] = operacion;
    montos[hilo] = (operacion == 2) ? 0 : cantidad;

    int nuevoSaldo = 0;

    if (operacion == 0) {
        nuevoSaldo = banco->depositar(cantidad);

        printf("[DEVICE] Bloque: %d | Hilo: %d | DEPOSITO | Monto: $%d | Nuevo saldo: $%d\n",
               blockIdx.x, threadIdx.x, cantidad, nuevoSaldo);
    }
    else if (operacion == 1) {
        nuevoSaldo = banco->retirar(cantidad);

        printf("[DEVICE] Bloque: %d | Hilo: %d | RETIRO | Monto: $%d | Saldo resultante: $%d\n",
               blockIdx.x, threadIdx.x, cantidad, nuevoSaldo);
    }
    else {
        nuevoSaldo = banco->consultarSaldo();

        printf("[DEVICE] Bloque: %d | Hilo: %d | CONSULTA | Monto: $0 | Saldo consultado: $%d\n",
               blockIdx.x, threadIdx.x, nuevoSaldo);
    }

    resultados[hilo] = nuevoSaldo;
}

// Integrantes:
// Guzman Lopez Paola Nicole
// Mondragon Gambino Juan Sinuhe
// Morales Limon Jennifer Nataly
class TestBanco {
public:
    void ejecutarPrimeraPrueba() {
        const int totalHilos = 100;
        const int saldoInicial = 10000;

        Banco* banco;
        int *montos, *operaciones, *resultados;

        cudaMallocManaged(&banco, sizeof(Banco));
        cudaMallocManaged(&montos, totalHilos * sizeof(int));
        cudaMallocManaged(&operaciones, totalHilos * sizeof(int));
        cudaMallocManaged(&resultados, totalHilos * sizeof(int));

        new (banco) Banco(saldoInicial);

        cout << "\n========================================\n";
        cout << "[HOST] PRIMERA EJECUCION CUDA\n";
        cout << "========================================\n";
        cout << "[HOST] Bloques: 1\n";
        cout << "[HOST] Hilos por bloque: 100\n";
        cout << "[HOST] Total de hilos: 100\n";
        cout << "[HOST] Saldo inicial: $" << saldoInicial << "\n\n";

        cudaEvent_t inicio, fin;
        cudaEventCreate(&inicio);
        cudaEventCreate(&fin);

        cudaEventRecord(inicio);

        operacionesAleatoriasKernel<<<1, totalHilos>>>(
            banco, montos, operaciones, resultados, 12345u
        );

        cudaEventRecord(fin);
        cudaEventSynchronize(fin);

        float tiempoMs = 0;
        cudaEventElapsedTime(&tiempoMs, inicio, fin);

        cout << "\n========================================\n";
        cout << "[HOST] RESULTADOS FINALES\n";
        cout << "========================================\n";
        cout << "[HOST] Saldo final: $" << banco->saldo << endl;
        cout << "[HOST] Tiempo del kernel: "
             << tiempoMs << " ms" << endl;

        cudaEventDestroy(inicio);
        cudaEventDestroy(fin);

        cudaFree(resultados);
        cudaFree(operaciones);
        cudaFree(montos);
        cudaFree(banco);
    }
};

int main() {
    cout << "Guzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly" << endl;

    TestBanco prueba;
    prueba.ejecutarPrimeraPrueba();

    cout << "\nGuzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly" << endl;
    return 0;
}


Overwriting banco.cu


In [ ]:

!nvcc -std=c++17 -arch=sm_75 banco.cu -o banco


In [ ]:

!./banco


/bin/bash: line 1: ./banco: No such file or directory


In [15]:

%%writefile banco.cu
#include <iostream>
#include <fstream>
#include <iomanip>
#include <new>
#include <cuda_runtime.h>

using namespace std;

#define VERIFICAR_CUDA(llamada) do {                         \
    cudaError_t error = (llamada);                          \
    if (error != cudaSuccess) {                             \
        cerr << "[ERROR CUDA] "                              \
             << cudaGetErrorString(error) << endl;          \
        exit(EXIT_FAILURE);                                 \
    }                                                       \
} while (0)

// Integrantes:
// Guzman Lopez Paola Nicole
// Mondragon Gambino Juan Sinuhe
// Morales Limon Jennifer Nataly
class Banco {
public:
    int saldo;
    int mutex;

    __host__ __device__
    Banco(int inicial = 10000) {
        saldo = inicial;
        mutex = 0;
    }

    __device__
    int depositar(int cantidad) {
        return atomicAdd(&saldo, cantidad) + cantidad;
    }

    __device__
    int retirar(int cantidad, int* aprobado) {
        int anterior;
        int observado = atomicAdd(&saldo, 0);

        do {
            anterior = observado;

            if (anterior < cantidad) {
                *aprobado = 0;
                return anterior;
            }

            observado = atomicCAS(
                &saldo, anterior, anterior - cantidad
            );

        } while (observado != anterior);

        *aprobado = 1;
        return anterior - cantidad;
    }

    __device__
    int consultarSaldo() {
        return atomicAdd(&saldo, 0);
    }
};

// Integrantes:
// Guzman Lopez Paola Nicole
// Mondragon Gambino Juan Sinuhe
// Morales Limon Jennifer Nataly
struct Registro {
    int operacion;
    int monto;
    int saldoResultante;
    int aprobado;
};

__device__
unsigned int generarAleatorio(unsigned int valor) {
    valor ^= valor << 13;
    valor ^= valor >> 17;
    valor ^= valor << 5;
    return valor;
}

// Kernel de deposito
__global__
void depositarKernel(Banco* banco, int cantidad, int* resultado) {
    if (blockIdx.x == 0 && threadIdx.x == 0)
        *resultado = banco->depositar(cantidad);
}

// Kernel de retiro
__global__
void retirarKernel(Banco* banco, int cantidad,
                   int* resultado, int* aprobado) {
    if (blockIdx.x == 0 && threadIdx.x == 0)
        *resultado = banco->retirar(cantidad, aprobado);
}

// Kernel de consulta
__global__
void consultarSaldoKernel(Banco* banco, int* resultado) {
    if (blockIdx.x == 0 && threadIdx.x == 0)
        *resultado = banco->consultarSaldo();
}

// Kernel principal: operaciones aleatorias concurrentes
__global__
void operacionesAleatoriasKernel(
    Banco* banco,
    Registro* registros,
    int totalHilos,
    unsigned int semilla,
    int imprimir
) {
    int globalIdx = blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx >= totalHilos)
        return;

    unsigned int aleatorio = generarAleatorio(
        semilla ^ ((globalIdx + 1u) * 747796405u)
    );

    int operacion = aleatorio % 3;

    aleatorio = generarAleatorio(aleatorio);
    int monto = (aleatorio % 500) + 1;

    int saldoNuevo = 0;
    int aprobado = 1;

    if (operacion == 0) {
        saldoNuevo = banco->depositar(monto);
    }
    else if (operacion == 1) {
        saldoNuevo = banco->retirar(monto, &aprobado);
    }
    else {
        monto = 0;
        saldoNuevo = banco->consultarSaldo();
    }

    registros[globalIdx].operacion = operacion;
    registros[globalIdx].monto = monto;
    registros[globalIdx].saldoResultante = saldoNuevo;
    registros[globalIdx].aprobado = aprobado;

    if (imprimir) {
        const char* nombre =
            operacion == 0 ? "DEPOSITO" :
            operacion == 1 ? "RETIRO" : "CONSULTA";

        printf(
            "[DEVICE] Bloque: %d | Hilo: %d | "
            "Global: %d | %s | Monto: $%d | "
            "Saldo: $%d | %s\n",
            blockIdx.x,
            threadIdx.x,
            globalIdx,
            nombre,
            monto,
            saldoNuevo,
            aprobado ? "OK" : "SIN FONDOS"
        );
    }
}

// Integrantes:
// Guzman Lopez Paola Nicole
// Mondragon Gambino Juan Sinuhe
// Morales Limon Jennifer Nataly
class TestBanco {
private:
    const int saldoInicial = 10000;

public:
    void ejecutar(int totalHilos, int hilosPorBloque,
                  bool guardarArchivo) {

        int bloques = (totalHilos + hilosPorBloque - 1)
                      / hilosPorBloque;

        Banco* banco = nullptr;
        Registro* registros = nullptr;

        VERIFICAR_CUDA(cudaMallocManaged(
            &banco, sizeof(Banco)
        ));

        VERIFICAR_CUDA(cudaMallocManaged(
            &registros, totalHilos * sizeof(Registro)
        ));

        new (banco) Banco(saldoInicial);

        cout << "\n========================================\n";
        cout << "[HOST] EJECUCION BANCARIA CUDA\n";
        cout << "========================================\n";
        cout << "[HOST] Bloques: " << bloques << endl;
        cout << "[HOST] Hilos por bloque: "
             << hilosPorBloque << endl;
        cout << "[HOST] Operaciones: "
             << totalHilos << endl;
        cout << "[HOST] Saldo inicial: $"
             << saldoInicial << endl;

        cudaEvent_t inicio, fin;

        VERIFICAR_CUDA(cudaEventCreate(&inicio));
        VERIFICAR_CUDA(cudaEventCreate(&fin));

        VERIFICAR_CUDA(cudaEventRecord(inicio));

        operacionesAleatoriasKernel<<<bloques, hilosPorBloque>>>(
            banco,
            registros,
            totalHilos,
            12345u,
            guardarArchivo ? 0 : 1
        );

        VERIFICAR_CUDA(cudaGetLastError());
        VERIFICAR_CUDA(cudaEventRecord(fin));
        VERIFICAR_CUDA(cudaEventSynchronize(fin));

        float tiempoMs = 0;

        VERIFICAR_CUDA(cudaEventElapsedTime(
            &tiempoMs, inicio, fin
        ));

        long long totalDepositado = 0;
        long long totalRetirado = 0;

        int depositos = 0;
        int retiros = 0;
        int consultas = 0;
        int rechazados = 0;

        ofstream archivo;

        if (guardarArchivo) {
            archivo.open("operaciones_bancarias_cuda.csv");

            if (!archivo.is_open()) {
                cerr << "[HOST] Error al crear archivo.\n";
                exit(EXIT_FAILURE);
            }

            archivo << "Bloque,Hilo,GlobalIdx,Operacion,"
                    << "Monto,SaldoResultante,Estado\n";
        }

        for (int i = 0; i < totalHilos; i++) {
            Registro r = registros[i];

            const char* nombre;
            const char* estado;

            if (r.operacion == 0) {
                nombre = "DEPOSITO";
                estado = "OK";
                depositos++;
                totalDepositado += r.monto;
            }
            else if (r.operacion == 1) {
                nombre = "RETIRO";
                retiros++;

                if (r.aprobado) {
                    estado = "OK";
                    totalRetirado += r.monto;
                }
                else {
                    estado = "SIN FONDOS";
                    rechazados++;
                }
            }
            else {
                nombre = "CONSULTA";
                estado = "OK";
                consultas++;
            }

            if (guardarArchivo) {
                archivo
                    << i / hilosPorBloque << ","
                    << i % hilosPorBloque << ","
                    << i << ","
                    << nombre << ","
                    << r.monto << ","
                    << r.saldoResultante << ","
                    << estado << "\n";
            }
        }

        if (guardarArchivo) {
            archivo.close();

            if (!archivo) {
                cerr << "[HOST] Error al escribir archivo.\n";
                exit(EXIT_FAILURE);
            }

            cout << "[HOST] Archivo generado: "
                 << "operaciones_bancarias_cuda.csv\n";
        }

        long long saldoEsperado =
            saldoInicial + totalDepositado - totalRetirado;

        cout << "\n========================================\n";
        cout << "[HOST] RESULTADOS FINALES\n";
        cout << "========================================\n";

        cout << "[HOST] Depositos: " << depositos << endl;
        cout << "[HOST] Retiros: " << retiros << endl;
        cout << "[HOST] Consultas: " << consultas << endl;
        cout << "[HOST] Retiros rechazados: "
             << rechazados << endl;

        cout << "[HOST] Total depositado: $"
             << totalDepositado << endl;

        cout << "[HOST] Total retirado: $"
             << totalRetirado << endl;

        cout << "[HOST] Saldo esperado: $"
             << saldoEsperado << endl;

        cout << "[HOST] Saldo final GPU: $"
             << banco->saldo << endl;

        cout << fixed << setprecision(4);
        cout << "[HOST] Tiempo del kernel: "
             << tiempoMs << " ms\n";

        if (saldoEsperado == banco->saldo) {
            cout << "[HOST] VERIFICACION: CORRECTA\n";
        }
        else {
            cout << "[HOST] VERIFICACION: ERROR\n";
        }

        VERIFICAR_CUDA(cudaEventDestroy(inicio));
        VERIFICAR_CUDA(cudaEventDestroy(fin));
        VERIFICAR_CUDA(cudaFree(registros));
        VERIFICAR_CUDA(cudaFree(banco));
    }

    void menu() {
        int opcion;

        do {
            cout << "\n========================================\n";
            cout << "       SISTEMA BANCARIO CUDA\n";
            cout << "========================================\n";
            cout << "1. Ejecutar 100 hilos\n";
            cout << "2. Ejecutar 1,000,000 de hilos\n";
            cout << "3. Salir\n";
            cout << "Seleccione una opcion: ";

            if (!(cin >> opcion)) {
                cin.clear();
                cin.ignore(10000, '\n');
                cout << "[HOST] Entrada invalida.\n";
                continue;
            }

            switch (opcion) {
                case 1:
                    ejecutar(100, 100, false);
                    break;

                case 2:
                    ejecutar(1000000, 256, true);
                    break;

                case 3:
                    cout << "[HOST] Finalizando programa.\n";
                    break;

                default:
                    cout << "[HOST] Opcion invalida.\n";
            }

        } while (opcion != 3);
    }
};

int main() {
    cout << "Guzman Lopez Paola Nicole | "
         << "Mondragon Gambino Juan Sinuhe | "
         << "Morales Limon Jennifer Nataly\n";

    TestBanco prueba;
    prueba.menu();

    cout << "Guzman Lopez Paola Nicole | "
         << "Mondragon Gambino Juan Sinuhe | "
         << "Morales Limon Jennifer Nataly\n";
    return 0;
}


Overwriting banco.cu


In [ ]:

!nvcc -std=c++17 -O2 -arch=sm_75 banco.cu -o banco


In [ ]:

import subprocess

resultado = subprocess.run(
    ["./banco"],
    input="2\n3\n",
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERROR:", resultado.stderr)


Guzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly

       SISTEMA BANCARIO CUDA
1. Ejecutar 100 hilos
2. Ejecutar 1,000,000 de hilos
3. Salir
Seleccione una opcion: 
[HOST] EJECUCION BANCARIA CUDA
[HOST] Bloques: 3907
[HOST] Hilos por bloque: 256
[HOST] Operaciones: 1000000
[HOST] Saldo inicial: $10000
[HOST] Archivo generado: operaciones_bancarias_cuda.csv

[HOST] RESULTADOS FINALES
[HOST] Depositos: 333594
[HOST] Retiros: 333019
[HOST] Consultas: 333387
[HOST] Retiros rechazados: 0
[HOST] Total depositado: $83677398
[HOST] Total retirado: $83539900
[HOST] Saldo esperado: $147498
[HOST] Saldo final GPU: $147498
[HOST] Tiempo del kernel: 1856.0076 ms
[HOST] VERIFICACION: CORRECTA

       SISTEMA BANCARIO CUDA
1. Ejecutar 100 hilos
2. Ejecutar 1,000,000 de hilos
3. Salir
Seleccione una opcion: [HOST] Finalizando programa.
Guzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly



In [ ]:

import os

archivo = "operaciones_bancarias_cuda.csv"

if os.path.exists(archivo):
    with open(archivo, "r") as f:
        total_lineas = sum(1 for linea in f)

    print("Archivo:", archivo)
    print("Registros:", total_lineas - 1)
    print("Tamaño (MB):", round(os.path.getsize(archivo) / 1048576, 2))

    print("\nPrimeras 5 operaciones:")
    with open(archivo, "r") as f:
        for _, linea in zip(range(6), f):
            print(linea.strip())
else:
    print("ERROR: No se generó el archivo.")


Archivo: operaciones_bancarias_cuda.csv
Registros: 1000000
Tamaño (MB): 35.94

Primeras 5 operaciones:
Bloque,Hilo,GlobalIdx,Operacion,Monto,SaldoResultante,Estado
0,0,0,CONSULTA,0,3190700,OK
0,1,1,CONSULTA,0,3190700,OK
0,2,2,RETIRO,67,2639454,OK
0,3,3,DEPOSITO,490,575786,OK
0,4,4,RETIRO,418,2623200,OK


In [ ]:

from google.colab import files

files.download("operaciones_bancarias_cuda.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import subprocess

while True:
    print("\n========================================")
    print("       SISTEMA BANCARIO CUDA")
    print("========================================")
    print("1. Ejecutar 100 hilos")
    print("2. Ejecutar 1,000,000 de hilos")
    print("3. Salir")

    opcion = input("\nSelecciona una opción: ").strip()

    if opcion not in ("1", "2", "3"):
        print("[HOST] Opción inválida.")
        continue

    if opcion == "3":
        print("[HOST] Finalizando menú.")
        break

    resultado = subprocess.run(
        ["./banco"],
        input=f"{opcion}\n3\n",
        text=True,
        capture_output=True
    )

    print(resultado.stdout)

    if resultado.returncode != 0:
        print("[ERROR]", resultado.stderr)



       SISTEMA BANCARIO CUDA
1. Ejecutar 100 hilos
2. Ejecutar 1,000,000 de hilos
3. Salir

Selecciona una opción: 1
Guzman Lopez Paola Nicole | Mondragon Gambino Juan Sinuhe | Morales Limon Jennifer Nataly

       SISTEMA BANCARIO CUDA
1. Ejecutar 100 hilos
2. Ejecutar 1,000,000 de hilos
3. Salir
Seleccione una opcion: 
[HOST] EJECUCION BANCARIA CUDA
[HOST] Bloques: 1
[HOST] Hilos por bloque: 100
[HOST] Operaciones: 100
[HOST] Saldo inicial: $10000
[DEVICE] Bloque: 0 | Hilo: 96 | Global: 96 | DEPOSITO | Monto: $311 | Saldo: $17199 | OK
[DEVICE] Bloque: 0 | Hilo: 98 | Global: 98 | CONSULTA | Monto: $0 | Saldo: $10000 | OK
[DEVICE] Bloque: 0 | Hilo: 99 | Global: 99 | DEPOSITO | Monto: $111 | Saldo: $17310 | OK
[DEVICE] Bloque: 0 | Hilo: 32 | Global: 32 | DEPOSITO | Monto: $491 | Saldo: $10491 | OK
[DEVICE] Bloque: 0 | Hilo: 33 | Global: 33 | CONSULTA | Monto: $0 | Saldo: $16987 | OK
[DEVICE] Bloque: 0 | Hilo: 34 | Global: 34 | CONSULTA | Monto: $0 | Saldo: $16987 | OK
[DEVICE] Bloque: 

In [ ]:

import csv
from collections import Counter

archivo = "operaciones_bancarias_cuda.csv"
conteo = Counter()
registros = 0

with open(archivo, newline="", encoding="utf-8") as f:
    lector = csv.DictReader(f)

    for fila in lector:
        registros += 1
        conteo[fila["Operacion"]] += 1

print("========================================")
print("VERIFICACION DEL ARCHIVO CUDA")
print("========================================")
print("Total de operaciones:", registros)
print("Depositos:", conteo["DEPOSITO"])
print("Retiros:", conteo["RETIRO"])
print("Consultas:", conteo["CONSULTA"])

if registros == 1000000:
    print("RESULTADO: CORRECTO")
else:
    print("RESULTADO: ERROR EN LA CANTIDAD DE REGISTROS")


VERIFICACION DEL ARCHIVO CUDA
Total de operaciones: 1000000
Depositos: 333594
Retiros: 333019
Consultas: 333387
RESULTADO: CORRECTO
